In [ ]:
!pip install pandas openpyxl

In [ ]:
import os

print(os.listdir())

['.config', 'title.basics.tsv.gz', 'title.principals.tsv.gz', 'title.ratings.tsv.gz', 'name.basics.tsv.gz', 'sample_data']


In [ ]:
import pandas as pd
import gc

print("Loading title.basics...")

basics = pd.read_csv(
    "title.basics.tsv.gz",
    sep="\t",
    compression="gzip",
    usecols=[
        "tconst",
        "titleType",
        "primaryTitle",
        "startYear",
        "genres"
    ],
    low_memory=False
)

movies = basics[
    basics["titleType"] == "movie"
].copy()

del basics
gc.collect()


print("Loading ratings...")

ratings = pd.read_csv(
    "title.ratings.tsv.gz",
    sep="\t",
    compression="gzip"
)

movies = movies.merge(
    ratings,
    on="tconst",
    how="inner"
)

del ratings
gc.collect()

movies = movies[
    movies["genres"] != "\\N"
]

movies = movies.sort_values(
    by=["averageRating", "numVotes"],
    ascending=False
)

movies = movies.head(500).copy()

movie_ids = set(movies["tconst"])

print("Selected movies:", len(movie_ids))

gc.collect()


print("Processing principals...")

principal_chunks = []

for chunk in pd.read_csv(
    "title.principals.tsv.gz",
    sep="\t",
    compression="gzip",
    usecols=["tconst", "nconst", "category"],
    chunksize=500000
):

    filtered = chunk[
        chunk["tconst"].isin(movie_ids)
    ]

    if not filtered.empty:
        principal_chunks.append(filtered)

principals = pd.concat(
    principal_chunks,
    ignore_index=True
)

del principal_chunks
gc.collect()

print("Principal rows:", len(principals))


needed_names = set(
    principals["nconst"]
)

print("Unique people:", len(needed_names))


print("Processing names...")

name_chunks = []

for chunk in pd.read_csv(
    "name.basics.tsv.gz",
    sep="\t",
    compression="gzip",
    usecols=["nconst", "primaryName"],
    chunksize=500000
):

    filtered = chunk[
        chunk["nconst"].isin(needed_names)
    ]

    if not filtered.empty:
        name_chunks.append(filtered)

names = pd.concat(
    name_chunks,
    ignore_index=True
)

del name_chunks
gc.collect()

print("Names loaded:", len(names))


directors = principals[
    principals["category"] == "director"
].copy()

directors = directors.merge(
    names,
    on="nconst",
    how="left"
)

director_map = (
    directors.groupby("tconst")
    ["primaryName"]
    .first()
    .to_dict()
)


actors = principals[
    principals["category"].isin(
        ["actor", "actress"]
    )
].copy()

actors = actors.merge(
    names,
    on="nconst",
    how="left"
)

actor_map = (
    actors.groupby("tconst")
    ["primaryName"]
    .apply(
        lambda x: ", ".join(
            x.dropna()
             .astype(str)
             .unique()[:5]
        )
    )
    .to_dict()
)


movies["Director"] = (
    movies["tconst"]
    .map(director_map)
)

movies["Top Actors"] = (
    movies["tconst"]
    .map(actor_map)
)


final_df = movies[
    [
        "primaryTitle",
        "startYear",
        "genres",
        "averageRating",
        "numVotes",
        "Director",
        "Top Actors"
    ]
].copy()

final_df.columns = [
    "Movie Name",
    "Release Year",
    "Genres",
    "Rating",
    "Votes",
    "Director",
    "Top Actors"
]


output_file = "IMDb_500_Movies_Full.xlsx"

final_df.to_excel(
    output_file,
    index=False
)

print("\nDataset created successfully.")
print("Rows:", len(final_df))
print("File:", output_file)

print("\nPreview:")
print(final_df.head())

Loading title.basics...
Loading ratings...
Selected movies: 500
Processing principals...
Principal rows: 6000
Unique people: 4583
Processing names...
Names loaded: 4583

Dataset created successfully.
Rows: 500
File: IMDb_500_Movies_Full.xlsx

Preview:
                               Movie Name Release Year       Genres  Rating  \
294980                         Sandigdham         2026       Action    10.0   
285079  A Black History Tour of St. Louis         2025  Documentary    10.0   
285361                        Ruby Island         2026     Thriller    10.0   
198002               The Story of Nehanda         2021      History    10.0   
284596      My boyfriends daddy is my man         2022        Adult    10.0   

        Votes               Director  \
294980   1000  Parda Saradhi Kommoju   
285079     11              Tony West   
285361     11          Freddy Moyano   
198002      7      Sydney Taivavashe   
284596      7            Jill Watson   

                                

In [ ]:
from google.colab import files

files.download("IMDb_500_Movies_Full.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>